In [ ]:
import os

os.chdir("../")

%pwd

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
from ultralytics import YOLO

In [ ]:
import os, cv2, csv, paddle
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from paddleocr import PaddleOCR


# Paths
MODEL_PATH = Path("runs/detect/anpr_v2/weights/best.pt")
IMG_DIR = Path("dataset/anpr/processed/images/test/")
OUTPUT_DIR = Path("results/anpr/paddleOCR/test/")
CSV_PATH = OUTPUT_DIR / "results.csv"
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png')

In [ ]:
# Load YOLO model
YOLO_MODEL = YOLO(MODEL_PATH)

# Load PaddleOCR
use_gpu = paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
device = 'gpu' if use_gpu else 'cpu'
print(f"[INFO] PaddleOCR will use: {device.upper()}")

OCR = PaddleOCR(use_textline_orientation=True, lang='en', use_gpu=use_gpu)  # English text OCR


def preprocess_plate(cropped):
    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    
    # Adaptive thresholding
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    
    # Resize for better recognition
    resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    processed_plate = cv2.cvtColor(resized, cv2.COLOR_GRAY2BGR)
    return processed_plate

def detect_and_ocr(image_path, yolo_model, paddle_ocr, output_dir):
    results_list = []
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            print(f"[WARN] Failed to read image: {image_path}")
            return

        results = yolo_model.predict(source=img, conf=0.5, save=False, verbose=False)

        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box[:4])
                cropped = img[y1:y2, x1:x2]

                try:
                    plate_text = ""
                    try:
                        processed_plate = preprocess_plate(cropped)
                        
                        # DEBUG: Save the processed plate before OCR
                        debug_crop_path = Path(output_dir) / f"debug_{Path(image_path).stem}_{i}.jpg"
                        cv2.imwrite(str(debug_crop_path), processed_plate)
    
                        # OCR on cropped plate
                        ocr_result = paddle_ocr.ocr(processed_plate)
                        
                        if ocr_result and isinstance(ocr_result, list) and len(ocr_result[0]) > 0:
                            plate_text = ocr_result[0][0][1][0]  # Text from first box
                        else:
                            print(f"[DEBUG] No text detected for {image_path}")
                    except Exception as ocr_err:
                        print(f"[ERROR] OCR failed at 01 for {image_path}: {ocr_err}")
                    
                    results_list.append((Path(image_path).name, plate_text))
                    
                    # Draw results
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(img, plate_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                    
                    print(f"[INFO] Detected Plate: {plate_text}")
                except Exception as ocr_err:
                    print(f"[ERROR] OCR failed at 02 for {image_path}: {ocr_err}")
        
        os.makedirs(output_dir, exist_ok=True)
        out_path = Path(output_dir) / Path(image_path).name
        
        cv2.imwrite(str(out_path), img)
        print(f"[INFO] Saved result: {out_path}")
        
        return results_list
    except Exception as e:
        print(f"[ERROR] Processing failed for {image_path}: {e}")

def save_results_to_csv(results, csv_path):
    os.makedirs(csv_path.parent, exist_ok=True)
    try:
        with open(csv_path, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["image_name", "detected_plate"])
            writer.writerows(results)
        print(f"[INFO] Results saved to {csv_path}")
    except Exception as e:
        print(f"[ERROR] Failed to save CSV: {e}")


def process_images_in_directory(img_dir=IMG_DIR, yolo_model=YOLO_MODEL, paddle_ocr=OCR, output_dir=OUTPUT_DIR, img_extensions=IMG_EXTENSIONS, csv_path=CSV_PATH):
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(img_extensions)]
    if not img_files:
        print(f"[WARN] No images found in {img_dir}")
        return

    all_results = []
    print(f"[INFO] Found {len(img_files)} images. Starting processing...")
    for img_file in tqdm(img_files, desc="Processing Images", unit="image"):
        img_path = os.path.join(img_dir, img_file)
        results = detect_and_ocr(img_path, yolo_model, paddle_ocr, output_dir)
        all_results.extend(results)

    save_results_to_csv(all_results, csv_path)
    print("[INFO] Processing completed.")


process_images_in_directory()